In [1]:
# 1. Write your custom function here!

# For this demo, we do a basic NDVI calculation, inputting and returning a pandas DataFrame.
# Note that the function must input and return a pandas DataFrame. This is a requirement.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df[["time", "X", "Y", "ndvi"]]

In [ ]:
import xee
help(xee.EarthEngineBackendEntrypoint)

In [5]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()


Successfully saved authorization token.


In [2]:
# 3. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

US_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/US")
Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
WSDemo_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")
Park_Lane_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/park_lane_tahoe")
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-12-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-12-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=Park_Lane_Boundaries.geometry(), crs='EPSG:3310', scale=30)

In [ ]:
print(ds)

In [ ]:
# 5. Preview the dataset and the results before doing a full run!
# This allows users to inspect the structure and content of the data to ensure it behaves as expected prior to running a full computation.
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': US_Boundaries,
    'crs': 'EPSG:3310',
    'scale': 30
},
    user_function=compute_ndvi,
    preview_dataset=True
)

In [3]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

df = pd.DataFrame(columns=["X", "Y", "SR_B4", "SR_B5", "ndvi"])
df_list = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}

run(
    dataset=ic,
    source="ee",
    preview_dataset = False,
    tune_function = True,
    max_pixels_per_tile = 40_000_000,
    dataset_config={
        'geometry': Park_Lane_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    function_tuning_config={
        #"chunks": chunks,
        "max_iterations": None
        #"output_template": df_list
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "PL_Tuned_NDVI_Tiles_30m_40MIL",
        "vrt": True,
        "report": True
    },
        #"target_blocks_per_run": 2000},
    dask_mode="custom",
    dask_config={
        "n_workers": 20,
        "threads_per_worker": 1,
        "memory_limit": "3GB",
        #"ee_max_concurrency": 30
    }
)
#[robustraster] ❌ ERROR during run(): Too Many Requests: Request was rejected because the request rate or concurrency limit was exceeded.
# EEException

r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\google\cloud\storage\_http.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:58847' processes=20 threads=20, memory=55.88 GiB>
[robustraster] Dask workers authenticated to Earth Engine.
[robustraster] Tuning user function...
AUTO
['time', 'X', 'Y']
[robustraster] ❌ ValueError during run(): Provided template has no dask arrays.  Please construct a template with appropriately chunked dask arrays.
[robustraster] Dask client closed.
[robustraster] Dask cluster shut down.


ValueError: Provided template has no dask arrays.  Please construct a template with appropriately chunked dask arrays.

In [ ]:
# Docker version
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': Plumas_Boundaries,
    'crs': 'EPSG:3310',
    'scale': 30
    },
    export_kwargs={
            "mode": "raster",
            "file_format": "GTiff",
            "output_folder": "tiles33"
    },
    dask_mode="custom",
    dask_kwargs={
            "n_workers": 4,
            "threads_per_worker": 1,
            "memory_limit": "1GB"
    },
    user_function=compute_ndvi,
    dask_use_docker=True,
    dask_docker_image="adrianomdocker/rr041"
)

df = pd.DataFrame(columns=["X", "Y", "SR_B4", "SR_B5", "ndvi"])
df_list = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': Park_Lane_Boundaries,
    'crs': 'EPSG:3310',
    'scale': 30,
    'tile_max_pixels': 40_000_000,
    "executor_kwargs": {"max_workers": 2}
    #'chunks': chunks
},
    user_function_params={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
        "chunks": chunks,
        "output_template": df_list,
        "max_iterations": None
    },
    export_kwargs={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Park_Lane_Docker_NDVI_Tiles_30m_40MIL",
        "vrt": True,
        "report": True},
        #"target_blocks_per_run": 2000},
    dask_mode="custom",
    dask_kwargs={
        "n_workers": 20,
        "threads_per_worker": 1,
        "memory_limit": "3GB",
        #"ee_max_concurrency": 30
    }
)

In [3]:
import time, socket, docker
from distributed import Client

def wait_port(host: str, port: int, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with socket.create_connection((host, port), timeout=1.5):
                return
        except OSError:
            time.sleep(0.5)
    raise TimeoutError(f"Port {host}:{port} not reachable in {timeout}s")

def connect_to_scheduler_from_host(scheduler_container_name: str, timeout=60):
    d = docker.from_env()
    c = d.containers.get(scheduler_container_name)
    # Fetch published host port (works even if Docker remapped to a random port)
    ports = c.attrs["NetworkSettings"]["Ports"]
    # Example: {'8786/tcp': [{'HostIp': '0.0.0.0', 'HostPort': '8786'}], ...}
    mapping = ports.get("8786/tcp")
    if not mapping:
        raise RuntimeError("Scheduler port 8786 is not published to host")
    host_ip  = mapping[0]["HostIp"] or "127.0.0.1"
    host_port = int(mapping[0]["HostPort"])
    host_ip = "127.0.0.1" if host_ip in ("0.0.0.0", "") else host_ip

    wait_port(host_ip, host_port, timeout=timeout)
    return Client(f"tcp://{host_ip}:{host_port}", timeout="60s")


In [ ]:
import multiprocessing
import docker
from dask.distributed import Client

cpu_cores = multiprocessing.cpu_count()

# harmonize kwargs
n_workers = 4
threads_per_worker = 1
memory_limit = "1GB"
#volumes = kwargs.get("volumes", {})
#ports = kwargs.get("ports", None)


################################################################################

workers = []

docker_client = docker.from_env()
scheduler = docker_client.containers.run(
    "adrianomdocker/rr041",
    command="dask-scheduler --host 0.0.0.0 --dashboard-address :8787",
    name="dask-scheduler",
    network="dask-network",
    detach=True,
    ports={'8786/tcp': 8786, '8787/tcp': 8787},
)
print(f"Dask Scheduler started: dask-scheduler ({scheduler.short_id})")

for i in range(1, n_workers + 1):
    name = f"worker{i}"
    worker = docker_client.containers.run(
            "adrianomdocker/rr041",
            command=f"dask-worker dask-scheduler:8786 --nthreads {threads_per_worker} --memory-limit {memory_limit}",
            name=name,
            network="dask-network",
            detach=True,
            mem_limit=memory_limit,
            #volumes=volumes or {},
        )
    workers.append(worker)
    print(f"Dask Worker {i} started: {name} ({worker.short_id})")
dask_client = connect_to_scheduler_from_host(scheduler_container_name="dask-scheduler")


In [6]:
import dask, distributed
print("dask:", dask.__version__)
print("distributed:", distributed.__version__)


dask: 2024.8.1
distributed: 2024.8.1
